# 1. Kết nối Google Drive và Giải nén Dataset
Khởi tạo môi trường, giải nén dữ liệu thô.


In [ ]:
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')
ZIP_PATH = '/content/drive/MyDrive/Tài liệu đồ án tốt nghiệp 2025.2/Dataset/archive.zip'
WORKSPACE = '/content/workspace'

if not os.path.exists(WORKSPACE):
    os.makedirs(WORKSPACE)
    !unzip -q "{ZIP_PATH}" -d {WORKSPACE}
    print("Giải nén hoàn tất!")
else:
    print("Workspace đã tồn tại!")


# 2. Tiền xử lý (Extract & Preprocess - Step 1)
Chuyển đổi gia tốc (g), góc quay (rad/s) và downsample 100Hz.


In [ ]:
import glob
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

INPUT_DIR = Path('/content/workspace/SisFall_dataset')
OUTPUT_DIR = Path('/content/workspace/SisFall_dataset_Processed')

target_har = {'D01', 'D02', 'D03', 'D04', 'D05', 'D07', 'D08', 'D11', 'D12', 'D13', 'D15', 'D17', 'D18', 'D19'}
ACCEL_SCALE = 32.0 / 8192.0
GYRO_SCALE = (4000.0 / 65536.0) * (np.pi / 180.0)

all_files = list(INPUT_DIR.rglob('*.txt'))
files_to_process = [f for f in all_files if f.name.startswith('F') or f.name[:3] in target_har]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for file_path in tqdm(files_to_process, desc="Preprocessing"):
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
            
        data = []
        for line in lines:
            line = line.strip().rstrip(';')
            if not line: continue
            parts = line.split(',')
            if len(parts) >= 6:
                data.append([float(x) for x in parts[:6]])
                
        if not data: continue
        df = pd.DataFrame(data, columns=['ax', 'ay', 'az', 'gx', 'gy', 'gz'])
        
        df[['ax', 'ay', 'az']] *= ACCEL_SCALE
        df[['gx', 'gy', 'gz']] *= GYRO_SCALE
        df = df.iloc[::2, :].reset_index(drop=True)
        
        rel_path = file_path.relative_to(INPUT_DIR)
        out_file = OUTPUT_DIR / rel_path.parent / (file_path.stem + '.csv')
        out_file.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_file, index=False)
    except Exception as e:
        pass
print("Hoàn tất Step 1!")


# 3. Cắt Window (Windowing - Step 2)
Áp dụng kỹ thuật cắt Peak và Sliding window.


In [ ]:
from scipy.signal import find_peaks

WINDOW_OUTPUT_DIR = Path('/content/workspace/SisFall_dataset_Windowed_new')

def extract_window(df, center, window_size=200):
    start, end = center - window_size // 2, center + window_size // 2
    if start < 0: start, end = 0, window_size
    if end > len(df): end, start = len(df), max(0, len(df) - window_size)
    return df.iloc[start:end]

def classify_idle(win_df):
    ax_m, ay_m, az_m = win_df['ax'].abs().mean(), win_df['ay'].abs().mean(), win_df['az'].abs().mean()
    return 'Idle_StandSit' if (ay_m > ax_m and ay_m > az_m) else 'Idle_Lie'

all_processed = list(OUTPUT_DIR.rglob('*.csv'))
trans_har = {'D07', 'D08', 'D11', 'D12', 'D13', 'D15', 'D17'}
cont_har = {'D01', 'D02', 'D03', 'D04', 'D05', 'D18', 'D19'}

windows_generated = 0
for f in tqdm(all_processed, desc="Windowing"):
    df = pd.read_csv(f)
    if len(df) < 200: continue
    
    filename, prefix = f.stem, f.stem[:3]
    out_dir = WINDOW_OUTPUT_DIR / f.parent.name
    out_dir.mkdir(parents=True, exist_ok=True)
    svm = np.sqrt(df['ax']**2 + df['ay']**2 + df['az']**2)
    
    if filename.startswith('F'):
        peak_idx = svm.argmax()
        for w_idx, shift in enumerate([-60, -30, 0, 30, 60]):
            win = extract_window(df, peak_idx + shift)
            if len(win) == 200:
                win.to_csv(out_dir / f"{filename}_Fall_W{w_idx:03d}.csv", index=False)
                windows_generated += 1
                
    elif prefix in trans_har:
        peaks, props = find_peaks(svm, height=1.0, distance=200)
        selected_peaks = np.sort(peaks[np.argsort(props['peak_heights'])[-2:]]) if len(peaks) >= 2 else peaks
        
        for i, peak in enumerate(selected_peaks):
            for shift_val in [-20, 0, 20]:
                win = extract_window(df, peak + shift_val)
                if len(win) == 200:
                    shift_name = "m20" if shift_val == -20 else "p20" if shift_val == 20 else "0"
                    win.to_csv(out_dir / f"{filename}_Trans_W{i:03d}_{shift_name}.csv", index=False)
                    windows_generated += 1
                
        for start in range(0, len(df) - 200 + 1, 100):
            center = start + 100
            if all(abs(center - p) >= 80 for p in selected_peaks):
                win = df.iloc[start:start+200]
                win.to_csv(out_dir / f"{filename}_{classify_idle(win)}_W{start//100:03d}.csv", index=False)
                windows_generated += 1
                
    elif prefix in cont_har:
        for start in range(0, len(df) - 200 + 1, 100):
            win = df.iloc[start:start+200]
            label = "Walk" if prefix in {'D01','D02','D05'} else "Run" if prefix in {'D03','D04'} else "Other"
            win.to_csv(out_dir / f"{filename}_{label}_W{start//100:03d}.csv", index=False)
            windows_generated += 1

print(f"Hoàn tất Windowing! Tổng số cửa sổ: {windows_generated}")


# 4. Chuẩn bị K-Fold và Import thư viện

In [ ]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import KFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from scipy import stats
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, SeparableConv1D, BatchNormalization, Activation, Multiply, Add, GlobalAveragePooling1D, GlobalMaxPooling1D, Concatenate, Dropout, Dense, Reshape
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Thư mục lưu kết quả. Ở đây lưu thẳng vào thư mục Dataset trên Drive của bạn.
DRIVE_OUT = Path('/content/drive/MyDrive/Tài liệu đồ án tốt nghiệp 2025.2/Dataset/KFold_Results')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print(f"[*] Kết quả sẽ được lưu tại: {DRIVE_OUT}")

# 5. K-Fold Data Manager
Load data và gom nhóm theo Subject ID để chia Subject-Independent.

In [ ]:
class KFoldDataManager:
    def __init__(self, windowed_dir, cache_dir, class_names):
        self.windowed_dir = Path(windowed_dir)
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.class_names = class_names
        
    def parse_filename_info(self, filename):
        if '_Trans_' in filename: return 'Trans'
        if '_StandSit_' in filename or '_Lie_' in filename: return 'Idle'
        prefix = filename[:3]
        if prefix in ['D01', 'D02', 'D05', 'D06']: return 'Walk'
        if prefix in ['D03', 'D04']: return 'Run'
        if prefix.startswith('F'): return 'Fall'
        return None

    def load_single_csv(self, file_path):
        try:
            filename = file_path.stem
            subject_id = filename.split('_')[1]
            label_str = self.parse_filename_info(filename)
            if label_str is None: return None
            label = self.class_names.index(label_str)
            df = pd.read_csv(file_path)
            if len(df) != 200: return None
            return df.to_numpy(), label, subject_id
        except: return None
        
    def load_and_group_by_subject(self):
        cache_file = self.cache_dir / 'subject_data.pkl'
        if cache_file.exists():
            print("[*] Loading subject data from cache...")
            with open(cache_file, 'rb') as f:
                return pickle.load(f)
                
        all_files = list(self.windowed_dir.rglob('*.csv'))
        if len(all_files) == 0:
            raise FileNotFoundError(f"LỖI: Không tìm thấy file CSV nào trong {self.windowed_dir}")
            
        subject_data = {}
        
        with ThreadPoolExecutor(max_workers=8) as executor:
            for res in tqdm(executor.map(self.load_single_csv, all_files), total=len(all_files), desc="Loading CSV"):
                if res is None: continue
                data, label, sub = res
                if sub not in subject_data:
                    subject_data[sub] = {'X': [], 'y': []}
                subject_data[sub]['X'].append(data)
                subject_data[sub]['y'].append(label)
                
        # Convert lists to numpy arrays
        for sub in subject_data:
            subject_data[sub]['X'] = np.array(subject_data[sub]['X'], dtype=np.float32)
            subject_data[sub]['y'] = np.array(subject_data[sub]['y'], dtype=np.int32)
            
        with open(cache_file, 'wb') as f:
            pickle.dump(subject_data, f)
            
        print(f"[*] Đã tải dữ liệu cho {len(subject_data)} subjects.")
        return subject_data

    def apply_preprocessing(self, X):
        if X.ndim == 3 and X.shape[0] > 0:
            X[:, :, 0:3] = np.clip(X[:, :, 0:3], -8.0, 8.0) / 8.0
            X[:, :, 3:6] = X[:, :, 3:6] / 2000.0
        return X

    def get_data_for_subjects(self, subject_data, subjects):
        X_list, y_list = [], []
        for sub in subjects:
            if sub in subject_data:
                X_list.append(subject_data[sub]['X'])
                y_list.append(subject_data[sub]['y'])
        if len(X_list) == 0:
            return np.array([]), np.array([])
        X = np.concatenate(X_list, axis=0)
        y = np.concatenate(y_list, axis=0)
        X = self.apply_preprocessing(X)
        return X, y

# 6. Kiến trúc Model (ResNet1D v25)

In [ ]:
def se_block_v25(x, c, ratio=4):
    se = GlobalAveragePooling1D()(x)
    se = Reshape((1, c))(se)
    se = Conv1D(c // ratio, kernel_size=1, use_bias=False)(se)
    se = Activation('relu6')(se)
    se = Conv1D(c, kernel_size=1, use_bias=False)(se)
    se = Activation('sigmoid')(se)
    return Multiply()([x, se])

def resnet1d_block(x, filters, kernel_size, strides, apply_se=True):
    residual = x
    y = SeparableConv1D(filters, kernel_size, strides=strides, padding='same', use_bias=False)(x)
    y = BatchNormalization()(y)
    y = Activation('relu6')(y)
    y = SeparableConv1D(filters, kernel_size, strides=1, padding='same', use_bias=False)(y)
    y = BatchNormalization()(y)
    
    if residual.shape[-1] != filters or strides != 1:
        residual = Conv1D(filters, kernel_size=1, strides=strides, padding='same', use_bias=False)(residual)
        residual = BatchNormalization()(residual)
        
    if apply_se:
        y = se_block_v25(y, c=filters, ratio=4)
        
    y = Add()([y, residual])
    y = Activation('relu6')(y)
    return Dropout(0.2)(y)

def build_model(input_shape=(200, 6), n_classes=5):
    inputs = Input(shape=input_shape)
    x = Conv1D(16, kernel_size=3, strides=2, padding='same', use_bias=False)(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu6')(x)
    
    x = resnet1d_block(x, filters=16, kernel_size=3, strides=1)
    x = resnet1d_block(x, filters=32, kernel_size=3, strides=2)
    x = resnet1d_block(x, filters=32, kernel_size=3, strides=1)
    x = resnet1d_block(x, filters=64, kernel_size=3, strides=2)
    
    gap = GlobalAveragePooling1D()(x)
    gmp = GlobalMaxPooling1D()(x)
    x = Concatenate()([gap, gmp])
    
    x = Dropout(0.3)(x)
    outputs = Dense(n_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), 
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1), 
        metrics=['accuracy']
    )
    return model

# 7. Hàm Train & Evaluate

In [ ]:
def augment_data(X, y):
    # 1. Augmentation chung: Scale random từ 0.95 đến 1.05 cho TOÀN BỘ dữ liệu
    scale_factors = np.random.uniform(0.95, 1.05, size=(X.shape[0], 1, 6)).astype(np.float32)
    X_aug = X * scale_factors
    
    X_list = [X, X_aug]
    y_list = [y, y]
    
    # 2. Augmentation biên độ cho nhãn Trans (Chỉ số index = 3)
    trans_idx = np.where(y == 3)[0]
    if len(trans_idx) > 0:
        X_trans = X[trans_idx]
        y_trans = y[trans_idx]
        
        # Chỉ gen thêm 2 biến thể về biên độ thôi vì thời gian đã được dịch từ CSV rồi
        X_trans_09 = X_trans * 0.9
        X_trans_11 = X_trans * 1.1
        
        X_list.extend([X_trans_09, X_trans_11])
        y_list.extend([y_trans, y_trans])
        
    # Gộp tất cả dữ liệu lại
    X_out = np.concatenate(X_list, axis=0)
    y_out = np.concatenate(y_list, axis=0)
    
    # Trộn ngẫu nhiên
    indices = np.arange(len(X_out))
    np.random.shuffle(indices)

def train_and_evaluate(X_train, y_train, X_test, y_test, run_name, class_names, epochs=50):
    print(f"\n{'='*50}\nRunning: {run_name}\n{'='*50}")
    print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")
    
    # Augment
    X_train, y_train = augment_data(X_train, y_train)
    
    # Class weights
    from sklearn.utils import class_weight
    if len(y_train) > 0:
        vals = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
        c_weights = dict(zip(np.unique(y_train), vals))
        if 4 in c_weights: # Fall class index
            c_weights[4] *= 3.0 
    else:
        c_weights = None
    
    y_train_oh = to_categorical(y_train, num_classes=5)
    y_test_oh = to_categorical(y_test, num_classes=5)
    
    model = build_model()
    
    model_path = DRIVE_OUT / f"model_{run_name}.keras"
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=0),
        ModelCheckpoint(filepath=str(model_path), monitor='val_loss', save_best_only=True, verbose=0)
    ]
    
    history = model.fit(
        X_train, y_train_oh,
        validation_split=0.1,
        epochs=epochs,
        batch_size=64,
        class_weight=c_weights,
        callbacks=callbacks,
        verbose=0
    )
    
    # Eval
    y_pred_probs = model.predict(X_test, batch_size=256, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    
    # Threshold cho Fall
    y_pred[y_pred_probs[:, 4] >= 0.25] = 4
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    
    # FIX: Thêm tham số labels=[0,1,2,3,4] để bắt buộc luôn map đủ 5 class
    all_labels = [0, 1, 2, 3, 4]
    report_dict = classification_report(
        y_test, y_pred, 
        labels=all_labels, 
        target_names=class_names, 
        output_dict=True, 
        zero_division=0
    )
    
    print(f"Accuracy: {acc:.4f} | F1-Score (Macro): {f1:.4f}")
    
    # FIX: Tương tự cho confusion_matrix
    cm = confusion_matrix(y_test, y_pred, labels=all_labels)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix: {run_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(DRIVE_OUT / f'CM_{run_name}.png')
    plt.close()
    
    # Cleanup mem
    tf.keras.backend.clear_session()
    
    return {'acc': float(acc), 'f1': float(f1), 'report': report_dict}


# 8. Cấu hình Dữ liệu và Các kịch bản K-Fold

In [ ]:
class_names = ['Walk', 'Run', 'Idle', 'Trans', 'Fall']
data_manager = KFoldDataManager('/content/workspace/SisFall_dataset_Windowed_new', '/content/workspace/cache', class_names)

# Tải dữ liệu và gom theo Subject
subject_data = data_manager.load_and_group_by_subject()

sa_subjects = [f"SA{i:02d}" for i in range(1, 24)]
se_subjects = [f"SE{i:02d}" for i in range(1, 16)]

# Lọc những subject thực tế có data
sa_subjects = [s for s in sa_subjects if s in subject_data]
se_subjects = [s for s in se_subjects if s in subject_data]

results = {}
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

### Kịch bản 1: Elderly Only (5-Fold)

In [ ]:
res_s1 = []
print("\n" + "="*50)
print("=== Scenario 1: Elderly Only (K-Fold) ===")
print("="*50)
for i, (train_idx, test_idx) in enumerate(kf.split(se_subjects)):
    train_subs = [se_subjects[idx] for idx in train_idx]
    test_subs = [se_subjects[idx] for idx in test_idx]
    
    X_tr, y_tr = data_manager.get_data_for_subjects(subject_data, train_subs)
    X_te, y_te = data_manager.get_data_for_subjects(subject_data, test_subs)
    
    res = train_and_evaluate(X_tr, y_tr, X_te, y_te, f"S1_Elderly_Fold{i+1}", class_names)
    res_s1.append(res)
    
results['S1_Elderly'] = res_s1

### Kịch bản 2: Young Only (5-Fold)

In [ ]:
res_s2 = []
print("\n" + "="*50)
print("=== Scenario 2: Young Only (K-Fold) ===")
print("="*50)
for i, (train_idx, test_idx) in enumerate(kf.split(sa_subjects)):
    train_subs = [sa_subjects[idx] for idx in train_idx]
    test_subs = [sa_subjects[idx] for idx in test_idx]
    
    X_tr, y_tr = data_manager.get_data_for_subjects(subject_data, train_subs)
    X_te, y_te = data_manager.get_data_for_subjects(subject_data, test_subs)
    
    res = train_and_evaluate(X_tr, y_tr, X_te, y_te, f"S2_Young_Fold{i+1}", class_names)
    res_s2.append(res)
    
results['S2_Young'] = res_s2

### Kịch bản 3: Train Elderly -> Test Young

In [ ]:
print("\n" + "="*50)
print("=== Scenario 3: Train Elderly -> Test Young ===")
print("="*50)
X_tr, y_tr = data_manager.get_data_for_subjects(subject_data, se_subjects)
X_te, y_te = data_manager.get_data_for_subjects(subject_data, sa_subjects)

res_s3 = train_and_evaluate(X_tr, y_tr, X_te, y_te, "S3_TrainSE_TestSA", class_names)
results['S3_TrainSE_TestSA'] = res_s3

### Kịch bản 4: Train Young -> Test Elderly

In [ ]:
print("\n" + "="*50)
print("=== Scenario 4: Train Young -> Test Elderly ===")
print("="*50)
X_tr, y_tr = data_manager.get_data_for_subjects(subject_data, sa_subjects)
X_te, y_te = data_manager.get_data_for_subjects(subject_data, se_subjects)

res_s4 = train_and_evaluate(X_tr, y_tr, X_te, y_te, "S4_TrainSA_TestSE", class_names)
results['S4_TrainSA_TestSE'] = res_s4

### Kịch bản 5: Train Both -> Test Both (5-Fold)

In [ ]:
res_s5 = []
print("\n" + "="*50)
print("=== Scenario 5: Train Both -> Test Both (K-Fold) ===")
print("="*50)
kf_sa = list(kf.split(sa_subjects))
kf_se = list(kf.split(se_subjects))

for i in range(n_splits):
    train_sa_idx, test_sa_idx = kf_sa[i]
    train_se_idx, test_se_idx = kf_se[i]
    
    train_subs = [sa_subjects[idx] for idx in train_sa_idx] + [se_subjects[idx] for idx in train_se_idx]
    test_subs = [sa_subjects[idx] for idx in test_sa_idx] + [se_subjects[idx] for idx in test_se_idx]
    
    X_tr, y_tr = data_manager.get_data_for_subjects(subject_data, train_subs)
    X_te, y_te = data_manager.get_data_for_subjects(subject_data, test_subs)
    
    res = train_and_evaluate(X_tr, y_tr, X_te, y_te, f"S5_Both_Fold{i+1}", class_names)
    res_s5.append(res)
    
results['S5_Both'] = res_s5

# 9. Statistical Significance Tests

In [ ]:
print("\n" + "="*50)
print("=== Statistical Significance Test ===")
print("="*50)

f1_s1 = [r['f1'] for r in results['S1_Elderly']]
f1_s2 = [r['f1'] for r in results['S2_Young']]
f1_s5 = [r['f1'] for r in results['S5_Both']]

# H1: Elderly vs Young (Independent T-test)
t_stat1, p_val1 = stats.ttest_ind(f1_s1, f1_s2)
print(f"H1 (Elderly vs Young F1): T-stat={t_stat1:.4f}, p-value={p_val1:.4f}")
if p_val1 < 0.05:
    print("-> Kết luận H1: Có sự khác biệt có ý nghĩa thống kê.")
else:
    print("-> Kết luận H1: Sự khác biệt KHÔNG có ý nghĩa thống kê.")

# H2: Both vs Elderly
t_stat2, p_val2 = stats.ttest_ind(f1_s5, f1_s1)
print(f"\nH2 (Both vs Elderly F1): T-stat={t_stat2:.4f}, p-value={p_val2:.4f}")
if p_val2 < 0.05:
    print("-> Kết luận H2: Cải thiện có ý nghĩa thống kê.")
else:
    print("-> Kết luận H2: Cải thiện KHÔNG có ý nghĩa thống kê.")

# Lưu kết quả tổng hợp vào file JSON
final_output = DRIVE_OUT / 'final_metrics.json'
with open(final_output, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)
    
print(f"\n[*] Đã lưu toàn bộ kết quả K-Fold và Metrics vào: {final_output}")